# Split-Step LSTM Classifier
Binary classifier: **1 = split step**, **0 = not a split step**.

In [ ]:
import json
import pathlib
import sys
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

sys.path.insert(0, "C:/Users/danie/OneDrive/Documents/GitHub/Split_Step_Counter")
from split_step.lstm_counter import SplitStepLSTM, N_POS, N_FEAT

## Configuration

In [ ]:
FEATURE_LANDMARK_NAMES = [
    "left_ankle",    "left_heel",     "left_foot_index",
    "right_ankle",   "right_heel",    "right_foot_index",
    "left_knee",     "right_knee",
    "left_hip",      "right_hip",
    "left_shoulder", "right_shoulder",
    "left_elbow",    "right_elbow",
    "left_wrist",    "right_wrist",
]
# Order must match LANDMARK_INDICES in split_step/lstm_counter.py — N_POS/N_FEAT
# are imported from there so the dimensions can never drift out of sync.

CHUNK_LEN        = 90     # frames per training chunk (~3s) — a batching
                           # convenience only; the model's actual context is
                           # unbounded since hidden state carries across chunks.
GAP_THRESHOLD_MS = 150    # a timestamp gap bigger than this (several dropped
                           # frames) starts a new sub-sequence instead of being
                           # bridged with a velocity computation. Must match
                           # _GAP_THRESHOLD_S in split_step/lstm_counter.py.
THRESHOLD        = 0.8
POS_WEIGHT_CAP   = 5.0

# Shared model artifact paths — must match split_step/paths.py so LSTMCounter
# picks up whatever this notebook trains without a manual copy step.
MODELS_DIR     = "C:/Users/danie/OneDrive/Documents/GitHub/Split_Step_Counter/models"
MODEL_PATH     = f"{MODELS_DIR}/split_step_lstm.pt"
FEAT_MEAN_PATH = f"{MODELS_DIR}/feat_mean.npy"
FEAT_STD_PATH  = f"{MODELS_DIR}/feat_std.npy"

print(f"Landmarks         : {len(FEATURE_LANDMARK_NAMES)}")
print(f"Position features : {N_POS}")
print(f"Model input size  : {N_FEAT}  (positions + velocities)")
print(f"Training chunk    : {CHUNK_LEN} frames  (~{CHUNK_LEN/30:.2f}s) — hidden state carries across chunks")
print(f"Gap threshold     : {GAP_THRESHOLD_MS}ms")
print(f"Threshold         : {THRESHOLD}")

## Load Training Datasets (multiple CSVs)

In [ ]:
def load_csv_groups(csv_path: str):
    """
    Load one CSV and split it into continuous groups.

    If the CSV has a 'source_video' column, each distinct value is its own
    group — so a sequence never spans two different source videos or a
    detected scene cut. Otherwise the whole file is treated as one
    continuous recording.

    Returns a list of (group_name, feature_frames, label_frames, timestamps_ms).
    """
    df = pd.read_csv(csv_path)
    feature_cols = [f"{name}_world_{ax}" for name in FEATURE_LANDMARK_NAMES for ax in ("x", "y", "z")]
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {missing}")
    if "label" not in df.columns:
        raise ValueError(f"'label' column not found in {csv_path} — run csv_labeler.py first.")

    df = df.dropna(subset=feature_cols)  # drop frames with no detected pose

    stem = pathlib.Path(csv_path).stem
    group_key = df["source_video"] if "source_video" in df.columns else pd.Series(stem, index=df.index)

    groups = []
    for group_name, g in df.groupby(group_key, sort=False):
        X  = g[feature_cols].values.astype(np.float32)
        y  = g["label"].values.astype(int).tolist()
        ts = g["timestamp_ms"].values.astype(float).tolist()
        groups.append((f"{stem}:{group_name}", [X[i] for i in range(len(X))], y, ts))
    return groups


TRAIN_CSV_PATHS = [
    "C:/Users/danie/OneDrive/Documents/GitHub/Split_Step_Counter/assets/csv_files/labeled_training_set_1.csv",
    "C:/Users/danie/OneDrive/Documents/GitHub/Split_Step_Counter/assets/csv_files/labeled_training_set_2.csv",
    "C:/Users/danie/OneDrive/Documents/GitHub/Split_Step_Counter/assets/csv_files/pose1_labeled.csv",
]

train_groups = []
for path in TRAIN_CSV_PATHS:
    train_groups.extend(load_csv_groups(path))

total    = sum(len(labels) for _, _, labels, _ in train_groups)
positive = sum(sum(labels) for _, _, labels, _ in train_groups)
print(f"Loaded {len(train_groups)} group(s), {total} frames total — {positive} positive ({positive/total*100:.1f}%)")
for name, feats, labels, _ in train_groups:
    print(f"  {name}: {len(labels)} frames, {sum(labels)} positive")

## Build Continuous Sequences (timestamp-aware velocity, gap-splitting)

In [ ]:
def build_continuous_sequences(
    feature_frames: List[np.ndarray],
    label_frames:   List[int],
    timestamps_ms:  List[float],
    gap_threshold_ms: float = GAP_THRESHOLD_MS,
) -> List[Tuple[np.ndarray, List[int]]]:
    """
    Turn one continuous group's per-frame data into one or more continuous
    sub-sequences of (position, velocity) feature vectors + labels.

    Velocity is a true timestamp-aware diff against the previous frame — not
    reset at an arbitrary window boundary like the old build_windows() did.
    A timestamp gap bigger than gap_threshold_ms (several frames dropped for
    low landmark visibility) starts a new sub-sequence instead of being
    bridged with a velocity computation.

    Returns a list of (features (T, N_FEAT), labels (length T)).
    """
    sub_sequences: List[Tuple[np.ndarray, List[int]]] = []
    cur_feats: List[np.ndarray] = []
    cur_labels: List[int] = []
    prev_pos = None
    prev_ts  = None

    for pos, label, ts in zip(feature_frames, label_frames, timestamps_ms):
        if prev_ts is not None and (ts - prev_ts) > gap_threshold_ms:
            if cur_feats:
                sub_sequences.append((np.stack(cur_feats), cur_labels))
            cur_feats, cur_labels = [], []
            prev_pos = None

        if prev_pos is None:
            vel = np.zeros_like(pos)
        else:
            dt_s = max((ts - prev_ts) / 1000.0, 1e-3)
            vel = (pos - prev_pos) / dt_s

        cur_feats.append(np.concatenate([pos, vel]))
        cur_labels.append(label)
        prev_pos = pos
        prev_ts  = ts

    if cur_feats:
        sub_sequences.append((np.stack(cur_feats), cur_labels))
    return sub_sequences


def iter_chunks(feats: np.ndarray, labels: List[int], chunk_len: int):
    """Yield non-overlapping (chunk_feats, chunk_labels) slices in order."""
    T = len(labels)
    for start in range(0, T, chunk_len):
        end = min(start + chunk_len, T)
        yield feats[start:end], labels[start:end]


def run_sequence(model, feats_n, labels, chunk_len, optimizer=None, criterion=None, device="cpu"):
    """
    Run one continuous sub-sequence through the model chunk-by-chunk,
    carrying LSTM hidden state between chunks (truncated BPTT). Trains if
    optimizer/criterion are given, otherwise just evaluates.

    Returns (loss_sum, n_frames, per_frame_logits).
    """
    hidden = None
    loss_sum = 0.0
    all_logits: List[float] = []

    for chunk_feats, chunk_labels in iter_chunks(feats_n, labels, chunk_len):
        x = torch.from_numpy(chunk_feats).unsqueeze(0).float().to(device)              # (1, T, N_FEAT)
        y = torch.tensor(chunk_labels, dtype=torch.float32).unsqueeze(0).to(device)    # (1, T)

        if hidden is not None:
            hidden = tuple(h.detach() for h in hidden)

        if optimizer is not None:
            optimizer.zero_grad()
            logits, hidden = model(x, hidden)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * x.shape[1]
        else:
            with torch.no_grad():
                logits, hidden = model(x, hidden)

        all_logits.extend(logits.detach().cpu().squeeze(0).tolist())

    return loss_sum, len(labels), all_logits

## Model

`SplitStepLSTM` is defined once, in `split_step/lstm_counter.py`, and imported
here — training and live inference share the exact same class, so they can
never drift apart (the old notebook/live-inference duplication was the root
cause of an earlier train/inference mismatch bug this project hit).

In [ ]:
print(SplitStepLSTM())

## Build Sequences + Train / Val Split (by sub-sequence, not by frame)

In [ ]:
# Build continuous sequences independently per group (so a sequence never
# spans two different source videos or a detected scene cut), then split
# whole sub-sequences (never mid-sequence) between train and val — splitting
# mid-sequence would break the point of carrying hidden state across chunks.
# The real held-out test set is loaded and evaluated separately further
# down, from test_set_labeled.csv — a video never seen during training.
all_sub_sequences = []  # (feats (T, N_FEAT) unnormalized, labels, group_name)
for name, feats, labels, ts in train_groups:
    for seq_feats, seq_labels in build_continuous_sequences(feats, labels, ts):
        all_sub_sequences.append((seq_feats, seq_labels, name))

rng = np.random.default_rng(42)
order = rng.permutation(len(all_sub_sequences))
shuffled = [all_sub_sequences[i] for i in order]

total_frames = sum(len(labels) for _, labels, _ in shuffled)
target_train_frames = int(total_frames * 0.85)

train_sequences, val_sequences = [], []
cum = 0
for seq in shuffled:
    if cum < target_train_frames:
        train_sequences.append(seq)
        cum += len(seq[1])
    else:
        val_sequences.append(seq)

train_frames = sum(len(l) for _, l, _ in train_sequences)
val_frames   = sum(len(l) for _, l, _ in val_sequences)
train_pos    = sum(sum(l) for _, l, _ in train_sequences)
val_pos      = sum(sum(l) for _, l, _ in val_sequences)

print(f"Total sub-sequences : {len(all_sub_sequences)}")
print(f"Train : {len(train_sequences)} sequences, {train_frames} frames ({train_pos} pos)")
print(f"Val   : {len(val_sequences)} sequences, {val_frames} frames ({val_pos} pos)")

## Feature Normalization

In [ ]:
# Fit on training data only — prevents validation/test statistics leaking into training
all_train_feats = np.concatenate([feats for feats, _, _ in train_sequences], axis=0)  # (train_frames, N_FEAT)
feat_mean = all_train_feats.mean(axis=0)
feat_std  = all_train_feats.std(axis=0) + 1e-8

def normalize_seq(feats):
    return (feats - feat_mean) / feat_std

pathlib.Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
np.save(FEAT_MEAN_PATH, feat_mean)
np.save(FEAT_STD_PATH,  feat_std)

print(f"Feature mean range : {feat_mean.min():.4f} – {feat_mean.max():.4f}")
print(f"Feature std range  : {feat_std.min():.4f} – {feat_std.max():.4f}")
print(f"Saved norm stats -> {FEAT_MEAN_PATH}, {FEAT_STD_PATH}")

## Training

In [ ]:
EPOCHS = 40
LR     = 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

n_pos = train_pos
n_neg = train_frames - train_pos
pw    = min(n_neg / n_pos, POS_WEIGHT_CAP)
pos_weight = torch.tensor([pw], dtype=torch.float32).to(device)
print(f"Train frames: {train_frames}  pos_weight: {pw:.2f}")

model     = SplitStepLSTM().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

epoch_rng = np.random.default_rng(0)
best_val_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    order = epoch_rng.permutation(len(train_sequences))  # shuffle sequence order each epoch;
                                                           # chunks within a sequence stay in order
    running_loss, running_n = 0.0, 0
    for i in order:
        feats, labels, _ = train_sequences[i]
        loss_sum, n, _ = run_sequence(model, normalize_seq(feats), labels, CHUNK_LEN,
                                       optimizer=optimizer, criterion=criterion, device=device)
        running_loss += loss_sum
        running_n    += n

    model.eval()
    tp = fp = tn = fn = 0
    for feats, labels, _ in val_sequences:
        _, _, logits = run_sequence(model, normalize_seq(feats), labels, CHUNK_LEN, device=device)
        preds = (torch.sigmoid(torch.tensor(logits)) >= THRESHOLD).int()
        y = torch.tensor(labels)
        tp += int(((preds == 1) & (y == 1)).sum())
        fp += int(((preds == 1) & (y == 0)).sum())
        tn += int(((preds == 0) & (y == 0)).sum())
        fn += int(((preds == 0) & (y == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    scheduler.step(1 - f1)
    print(f"Epoch {epoch:3d}/{EPOCHS}  loss={running_loss/running_n:.4f}  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}")

    if f1 > best_val_f1:
        best_val_f1 = f1
        pathlib.Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), MODEL_PATH)

print(f"\nBest val F1={best_val_f1:.3f} — model saved -> {MODEL_PATH}")

## Held-Out Test Evaluation (test_set_labeled.csv — unseen video)

In [ ]:
TEST_CSV_PATH = "C:/Users/danie/OneDrive/Documents/GitHub/Split_Step_Counter/assets/csv_files/test_set_labeled.csv"

test_groups = load_csv_groups(TEST_CSV_PATH)

test_sequences = []  # (feats, labels, timestamps_ms) — timestamps aligned 1:1 with frames now
for name, feats, labels, ts in test_groups:
    offset = 0
    for seq_feats, seq_labels in build_continuous_sequences(feats, labels, ts):
        seq_ts = ts[offset: offset + len(seq_labels)]
        offset += len(seq_labels)
        test_sequences.append((seq_feats, seq_labels, seq_ts))
    print(f"  {name}: {len(labels)} frames, {sum(labels)} positive")

model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

all_probs, all_gt, all_ts = [], [], []
for feats, labels, ts in test_sequences:
    _, _, logits = run_sequence(model, normalize_seq(feats), labels, CHUNK_LEN, device=device)
    all_probs.extend(torch.sigmoid(torch.tensor(logits)).tolist())
    all_gt.extend(labels)
    all_ts.extend(ts)

P = (torch.tensor(all_probs) >= THRESHOLD).int()
G = torch.tensor(all_gt)
tp = int(((P==1)&(G==1)).sum())
fp = int(((P==1)&(G==0)).sum())
tn = int(((P==0)&(G==0)).sum())
fn = int(((P==0)&(G==1)).sum())

precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
recall    = tp/(tp+fn) if (tp+fn)>0 else 0.0
f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0

print(f"\nHeld-out test set ({len(G)} frames)")
print(f"  Accuracy  : {(tp+tn)/len(G):.3f}")
print(f"  Precision : {precision:.3f}")
print(f"  Recall    : {recall:.3f}")
print(f"  F1        : {f1:.3f}")
print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")

## Prediction vs Ground Truth Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SAMPLE_MS = 100 / 3  # 33.33ms per frame

all_probs_arr = np.array(all_probs)
all_gt_arr    = np.array(all_gt)
timestamps    = np.array(all_ts)  # absolute ms in the original held-out video, 1:1 with predictions

def frame_label(ts_ms):
    frame     = round(ts_ms / SAMPLE_MS)
    sets      = frame // 30
    remaining = frame % 30
    return f"{sets}:{remaining:02d}"

fig, ax = plt.subplots(figsize=(18, 4))

in_block = False
for i, gt in enumerate(all_gt_arr):
    if gt == 1 and not in_block:
        block_start, in_block = timestamps[i], True
    elif gt == 0 and in_block:
        ax.axvspan(block_start, timestamps[i], color="green", alpha=0.25)
        in_block = False
if in_block:
    ax.axvspan(block_start, timestamps[-1], color="green", alpha=0.25)

ax.plot(timestamps, all_probs_arr, color="steelblue", linewidth=0.8)
ax.axhline(THRESHOLD, color="red", linestyle="--", linewidth=1)

mask    = all_probs_arr >= THRESHOLD
pred_ts = timestamps[mask]
pred_p  = all_probs_arr[mask]
ax.scatter(pred_ts, pred_p, color="red", s=20, zorder=5)
for ts, p in zip(pred_ts, pred_p):
    ax.annotate(frame_label(ts), xy=(ts, p), xytext=(0, 6),
                textcoords="offset points", ha="center", fontsize=6, color="red")

ax.set_xlim(timestamps[0], timestamps[-1])
ax.set_ylim(0, 1.1)
ax.set_xlabel("Time (ms, in original held-out video)")
ax.set_ylabel("Predicted probability")
ax.set_title("Held-out test set: model predictions vs ground truth (green = actual split step)")
ax.legend(handles=[
    mpatches.Patch(color="green", alpha=0.4, label="Ground truth"),
    plt.Line2D([0],[0], color="steelblue", label="Predicted probability"),
    plt.Line2D([0],[0], color="red", linestyle="--", label=f"Threshold ({THRESHOLD})"),
    plt.Line2D([0],[0], marker="o", color="red", linestyle="None", label="Predicted split step"),
])
plt.tight_layout()
plt.show()

## Live Inference

No separate `Predictor` class here anymore — `split_step.lstm_counter.LSTMCounter`
*is* the streaming predictor (stateful, one frame at a time, no rolling
window buffer). Re-defining similar inference logic in the notebook is
exactly the duplication that caused problems before, so the app and any
notebook experimentation both import the same class.

In [ ]:
from split_step.lstm_counter import LSTMCounter

counter = LSTMCounter()  # loads the model/norm stats just trained and saved above
print(f"Loaded LSTMCounter — count={counter.count}, state={counter.state}")

## Persistence (JSON)

In [38]:
def save_dataset(feature_frames, label_frames, path):
    pathlib.Path(path).write_text(json.dumps(
        {"features": [f.tolist() for f in feature_frames], "labels": label_frames}))

def load_dataset(path):
    payload = json.loads(pathlib.Path(path).read_text())
    return [np.array(f, dtype=np.float32) for f in payload["features"]], payload["labels"]